In [1]:
import json
import asyncio
import re
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse, parse_qs

from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError
from bs4 import BeautifulSoup


# =========================
# Config (분쟁조정사례 116)
# =========================
BASE = "https://www.consumer.go.kr"

LIST_URL_TMPL = (
    "https://www.consumer.go.kr/user/ftc/consumer/trublmdatcase/116/selectTrublMdatCaseList.do"
    "?page={page}&row=25&searchType=&searchCnd=&searchWrd="
)

OUT_JSONL = "trubl_mdat_cases_116_full.jsonl"
ERROR_JSONL = "trubl_mdat_cases_116_errors.jsonl"

HEADLESS = True
TIMEOUT_MS = 30_000

MAX_RETRIES = 3
RETRY_BACKOFF_SEC = 1.5
CHECKPOINT_EVERY = 1

# selectors
LIST_ROW_SELECTOR = "table.tbl.col.data tbody tr"
DETAIL_TABLE_SELECTOR = "table.tbl.row.data"

# 병렬 상세 처리 개수(안전)
DETAIL_CONCURRENCY = 3


# =========================
# Utils
# =========================
def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def normalize_text(text: str) -> str:
    """
    RAG용 '보수적' 정규화:
    - 연속 줄바꿈(3개 이상)만 2개로 축소
    - 탭/과한 공백 정리
    """
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    # html에서 <p><br></p> 같은게 줄바꿈으로 많이 들어올 수 있어 보수적으로 정리
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def safe_text(el) -> str:
    if not el:
        return ""
    return normalize_text(el.get_text("\n", strip=True))

def extract_case_sn(url: str) -> str | None:
    # ✅ 116 상세 파라미터명: trublMdatCaseSn
    qs = parse_qs(urlparse(url).query)
    v = qs.get("trublMdatCaseSn")
    return v[0] if v else None

def make_doc_id(url: str) -> str:
    sn = extract_case_sn(url)
    # ✅ doc_type prefix로 유니크 보장(상담/조정/법령 섞여도 충돌 방지)
    return f"mediation:{sn}" if sn else f"mediation:url:{url}"


# =========================
# Parsing: 상세
# =========================
def parse_detail_html(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    def get_row_exact(label: str) -> str:
        """
        <th>라벨</th><td>값</td> 형태에서 '정확히' 라벨 매칭.
        (label in text 방식보다 오매칭 줄이려고 exact로)
        """
        for th in soup.select("table.tbl.row.data th"):
            th_txt = th.get_text(strip=True)
            if th_txt == label:
                td = th.find_next_sibling("td")
                if not td:
                    return ""
                return safe_text(td)
        return ""

    title = get_row_exact("제목")
    source = get_row_exact("출처")
    category = get_row_exact("분류")
    views = get_row_exact("조회수")

    summary = get_row_exact("사건개요")
    claims = get_row_exact("당사자 주장")
    reasoning = get_row_exact("판단")
    decision = get_row_exact("결정사항")

    # RAG용 content 구성(문서 유형 통일 포맷)
    parts = []
    parts.append("[문서유형] 분쟁조정 사례")
    if title: parts.append(f"[제목] {title}")
    if source: parts.append(f"[출처] {source}")
    if category: parts.append(f"[분류] {category}")
    if views: parts.append(f"[조회수] {views}")

    body = []
    if summary: body.append(f"[사건개요]\n{summary}")
    if claims: body.append(f"[당사자 주장]\n{claims}")
    if reasoning: body.append(f"[판단]\n{reasoning}")
    if decision: body.append(f"[결정사항]\n{decision}")

    if body:
        parts.append("\n".join(body))

    content = "\n\n".join(parts).strip()

    return {
        "id": make_doc_id(url),
        "url": url,
        "title": title,
        "source": source,
        "category": category,
        "views": views,
        "summary": summary,        # 사건개요
        "claims": claims,          # 당사자 주장
        "reasoning": reasoning,    # 판단
        "decision": decision,      # 결정사항
        "content": content,
        "collected_at": now_iso(),
        "metadata": {
            "site": "consumer.go.kr",
            "doc_type": "consumer_mediation_case",
            "case_sn": extract_case_sn(url),
        },
    }


# =========================
# IO
# =========================
def load_seen_ids(path: str) -> set[str]:
    seen = set()
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    _id = obj.get("id")
                    if _id:
                        seen.add(str(_id))
                except:
                    pass
    except FileNotFoundError:
        pass
    return seen

def append_jsonl(fp, obj: dict):
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")
    fp.flush()


# =========================
# Network optimization
# =========================
async def block_heavy_assets(route):
    rtype = route.request.resource_type
    # ✅ 안정성 우선: stylesheet 차단이 문제 생기면 여기서 "stylesheet"만 빼면 됨
    if rtype in ("image", "font", "media", "stylesheet"):
        await route.abort()
    else:
        await route.continue_()


# =========================
# 목록 메타 추출 (116 버전)
# =========================
async def extract_list_items(list_page) -> list[dict]:
    """
    116 목록 tr 단위:
    - link href
    - 번호, 제목, 출처, 조회수
    """
    items = await list_page.locator(LIST_ROW_SELECTOR).evaluate_all(
        """rows => rows.map(tr => {
            const getText = (sel) => {
                const el = tr.querySelector(sel);
                return el ? el.textContent.trim() : "";
            };

            const a = tr.querySelector("td.title a");
            const href = a ? a.getAttribute("href") : "";

            return {
                href,
                no: getText('td[aria-label="번호"]'),
                title_list: getText('td[aria-label="제목"]'),
                source_list: getText('td[aria-label="출처"]'),
                views_list: getText('td[aria-label="조회수"]'),
            };
        })"""
    )

    cleaned = []
    for it in items:
        href = (it.get("href") or "").strip()
        if not href:
            continue
        it["url"] = urljoin(BASE, href)
        it["case_sn"] = extract_case_sn(it["url"])
        cleaned.append(it)
    return cleaned


# =========================
# Main
# =========================
async def main(start_page=1, end_page=50):
    seen = load_seen_ids(OUT_JSONL)
    print("seen already:", len(seen))

    sem = asyncio.Semaphore(DETAIL_CONCURRENCY)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
            )
        )
        context.set_default_timeout(TIMEOUT_MS)
        await context.route("**/*", block_heavy_assets)

        list_page = await context.new_page()

        saved = 0
        skipped = 0
        failed = 0

        with open(OUT_JSONL, "a", encoding="utf-8") as out_fp, \
             open(ERROR_JSONL, "a", encoding="utf-8") as err_fp:

            async def fetch_detail(url: str, page_no: int, list_meta: dict):
                nonlocal saved, skipped, failed

                async with sem:
                    doc_id = make_doc_id(url)
                    if doc_id in seen:
                        skipped += 1
                        return

                    detail_page = await context.new_page()
                    try:
                        last_err = None
                        for attempt in range(1, MAX_RETRIES + 1):
                            try:
                                await detail_page.goto(url, wait_until="domcontentloaded")
                                await detail_page.wait_for_selector(DETAIL_TABLE_SELECTOR)

                                html = await detail_page.content()
                                item = parse_detail_html(html, url)

                                # 최소 검증(제목/결정사항/사건개요 중 하나라도 있어야 정상)
                                if not (item["title"] or item["decision"] or item["summary"]):
                                    raise RuntimeError("empty parsed fields")

                                # ✅ 목록 메타 병합
                                item["list_meta"] = {
                                    "list_page": page_no,
                                    "no": list_meta.get("no", ""),
                                    "title_list": list_meta.get("title_list", ""),
                                    "source_list": list_meta.get("source_list", ""),
                                    "views_list": list_meta.get("views_list", ""),
                                }

                                # ✅ metadata에도 필터 가능한 값 복사(검색용)
                                item["metadata"].update({
                                    "views": (item.get("views") or list_meta.get("views_list", "")).strip(),
                                    "source_list": list_meta.get("source_list", "").strip(),
                                    "category": (item.get("category") or "").strip(),
                                })

                                append_jsonl(out_fp, item)
                                seen.add(item["id"])
                                saved += 1
                                return

                            except Exception as e:
                                last_err = e
                                if attempt < MAX_RETRIES:
                                    await asyncio.sleep(RETRY_BACKOFF_SEC * attempt)
                                else:
                                    failed += 1
                                    append_jsonl(err_fp, {
                                        "url": url,
                                        "id": doc_id,
                                        "page": page_no,
                                        "error": repr(last_err),
                                        "at": now_iso(),
                                    })
                    finally:
                        await detail_page.close()

            for pg in range(start_page, end_page + 1):
                list_url = LIST_URL_TMPL.format(page=pg)

                list_items = []
                for attempt in range(1, MAX_RETRIES + 1):
                    try:
                        await list_page.goto(list_url, wait_until="domcontentloaded")
                        await list_page.wait_for_selector(LIST_ROW_SELECTOR)
                        list_items = await extract_list_items(list_page)

                        if len(list_items) == 0:
                            raise RuntimeError("list empty (0 rows with href)")

                        break

                    except (PWTimeoutError, Exception) as e:
                        if attempt < MAX_RETRIES:
                            await asyncio.sleep(RETRY_BACKOFF_SEC * attempt)
                        else:
                            tr_cnt = await list_page.locator("tbody tr").count()
                            print(f"[list][fail] page={pg} tr={tr_cnt} err={repr(e)}")
                            append_jsonl(err_fp, {
                                "url": list_url,
                                "page": pg,
                                "error": f"list_page_failed: {repr(e)}",
                                "at": now_iso(),
                            })
                            list_items = []

                if not list_items:
                    continue

                print(f"[list] page={pg} items={len(list_items)}")

                tasks = []
                for meta in list_items:
                    tasks.append(fetch_detail(meta["url"], pg, meta))

                await asyncio.gather(*tasks)

                if pg % CHECKPOINT_EVERY == 0:
                    print(f"✅ checkpoint page {pg} | saved={saved} skipped={skipped} failed={failed}")

        await browser.close()

    print("\n==== DONE ====")
    print(f"pages: {start_page}~{end_page}")
    print(f"saved: {saved}")
    print(f"skipped(seen): {skipped}")
    print(f"failed: {failed}")


# 실행 예시
await main(start_page=1, end_page=100)

seen already: 0
[list] page=1 items=25
✅ checkpoint page 1 | saved=25 skipped=0 failed=0
[list] page=2 items=25
✅ checkpoint page 2 | saved=50 skipped=0 failed=0
[list] page=3 items=25
✅ checkpoint page 3 | saved=75 skipped=0 failed=0
[list] page=4 items=25
✅ checkpoint page 4 | saved=100 skipped=0 failed=0
[list] page=5 items=25
✅ checkpoint page 5 | saved=125 skipped=0 failed=0
[list] page=6 items=25
✅ checkpoint page 6 | saved=150 skipped=0 failed=0
[list] page=7 items=25
✅ checkpoint page 7 | saved=175 skipped=0 failed=0
[list] page=8 items=25
✅ checkpoint page 8 | saved=200 skipped=0 failed=0
[list] page=9 items=25
✅ checkpoint page 9 | saved=225 skipped=0 failed=0
[list] page=10 items=25
✅ checkpoint page 10 | saved=250 skipped=0 failed=0
[list] page=11 items=25
✅ checkpoint page 11 | saved=275 skipped=0 failed=0
[list] page=12 items=25
✅ checkpoint page 12 | saved=300 skipped=0 failed=0
[list] page=13 items=25
✅ checkpoint page 13 | saved=325 skipped=0 failed=0
[list] page=14 it